In [1]:
# 02_preprocessing_and_sequencing

#Preprocess numeric features (forward-fill + MinMax scaling), build sliding-window sequences (X,y), and save train/test arrays.


In [2]:
# Setup
import os, sys
project_root = r"C:\Users\shiva\Documents\multicommodity-prices-indonesia"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
from pathlib import Path
import json

# Import helper code from src if available
from src.data import loader, preprocess, sequence_builder
print("Modules imported.")


SyntaxError: unexpected character after line continuation character (loader.py, line 1)

In [ ]:
csv_path = Path(project_root)/"data"/"csv"/"india_commodities_demo.csv"
if not csv_path.exists():
    # fallback: find any csv
    csvs = list((Path(project_root)/"data"/"csv").glob("*.csv"))
    if len(csvs)==0:
        raise FileNotFoundError("No CSV found under data/csv")
    csv_path = csvs[0]
print("Loading:", csv_path)
df = pd.read_csv(csv_path, index_col=0, parse_dates=True)
print("Loaded shape:", df.shape)
df = df.select_dtypes(include=['number']).copy()  # keep numeric
cols = df.columns.tolist()
print("Columns:", cols)


In [ ]:
# Preprocess and normalize
arr_norm, scaler = preprocess.fit_transform(df)
print("Normalized array shape:", arr_norm.shape)
# save scaler to disk
scaler_path = Path(project_root)/"data"/"processed"/"scaler.json"
Path(scaler_path).parent.mkdir(parents=True, exist_ok=True)
with open(scaler_path, "w", encoding="utf8") as f:
    json.dump(scaler, f, indent=2)
print("Saved scaler to:", scaler_path)


In [ ]:
seq_len = 30  # as used in the paper
X, y = sequence_builder.build_sequences(arr_norm, seq_len=seq_len)
print("Built sequences X,y shapes:", X.shape, y.shape)

# time-based split: first 80% for train, last 20% for test
N = X.shape[0]
split = int(N * 0.8)
X_train, y_train = X[:split], y[:split]
X_test, y_test = X[split:], y[split:]
print("Train/test sizes:", X_train.shape[0], X_test.shape[0])

# Save to processed folder
proc = Path(project_root)/"data"/"processed"
proc.mkdir(parents=True, exist_ok=True)
np.save(proc/"X_train.npy", X_train)
np.save(proc/"y_train.npy", y_train)
np.save(proc/"X_test.npy", X_test)
np.save(proc/"y_test.npy", y_test)
print("Saved processed arrays to", proc)


In [ ]:
# Sanity inspect one sample (denormalize to original units for readability)
with open(scaler_path, "r", encoding="utf8") as f:
    loaded_scaler = json.load(f)

def denorm_one(x_norm, cols, scaler_dict):
    mins = np.array([scaler_dict[c][0] for c in cols], dtype=float)
    maxs = np.array([scaler_dict[c][1] for c in cols], dtype=float)
    ranges = (maxs - mins)
    ranges[ranges==0] = 1.0
    return x_norm * ranges + mins

sample_idx = 0
sample_X = X_train[sample_idx]   # (seq_len, features)
sample_y = y_train[sample_idx]   # (features,)
sample_X_denorm = denorm_one(sample_X, cols, loaded_scaler)
sample_y_denorm = denorm_one(sample_y, cols, loaded_scaler)
print("Sample X (last 3 timesteps) denormalized:")
display(pd.DataFrame(sample_X_denorm[-3:, :], columns=cols))
print("Target (next step) denormalized:")
display(pd.Series(sample_y_denorm, index=cols))
